# 02 · Seçilen Yılların Ön İşlenmesi (Colab'da çalıştır)

`configs/data.json` dosyasındaki her yıl için:

1. arşiv Zenodo'dan Colab diskine indirilir (15–18 GB, geçici),
2. her örnek **arşiv açılmadan** okunur; DBZ + VEL, en alçak tarama ve son zaman adımı alınıp `uint8` olarak saklanır,
3. `tornet_<yıl>.npz` Google Drive'a kaydedilir (yıl başına yaklaşık 1–1,5 GB),
4. arşiv silinir.

**Colab'da CPU çalışma zamanıyla çalıştır.** GPU gerekmez, GPU kotan da harcanmaz.
Biraz sürer (yıl başına yaklaşık 15–30 dakika, çoğu indirme). Bağlantı koparsa defteri tekrar çalıştır:
bitmiş yıllar atlanır, yarım kalan indirme kaldığı yerden devam eder.

Google Drive'da yaklaşık 4–5 GB boş yer gerekir.

In [ ]:
# --- Kurulum: hem Google Colab'da hem kendi bilgisayarında çalışır ---
import os, sys
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive, userdata
    drive.mount("/content/drive")                     # işlenmiş veri Drive'da saklanır
    REPO = Path("/content/tornado-cnn")
    if not REPO.exists():
        try:                                          # repo private ise: Colab Secrets'taki token
            token = userdata.get("GITHUB_TOKEN") + "@"
        except Exception:                             # repo public ise token gerekmez
            token = ""
        !git clone -q https://{token}github.com/baevv/tornado-cnn.git {REPO}
    else:
        !git -C {REPO} pull -q
    %pip install -q h5netcdf h5py
    os.chdir(REPO)
else:
    REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

sys.path.insert(0, str(REPO))
print("Colab" if IN_COLAB else "Yerel", "| repo:", REPO)

In [ ]:
import shutil
import numpy as np
from src import paths, tornet

cfg = paths.load_config()
SCRATCH = paths.scratch_dir()      # geçici, büyük
OUT = paths.processed_dir()        # kalıcı (Colab'da Drive)

def free_gb(p):
    return shutil.disk_usage(p).free / 1e9

print("yıllar:", cfg["years"])
print(f"geçici klasör: {SCRATCH}  ({free_gb(SCRATCH):.0f} GB boş)")
print(f"çıktı klasörü: {OUT}  ({free_gb(OUT):.0f} GB boş)")

In [ ]:
cat = tornet.get_catalog(paths.data_dir())

for year in cfg["years"]:
    out_file = OUT / f"tornet_{year}.npz"
    if out_file.exists():
        print(f"{year}: zaten tamamlanmış")
        continue
    tar_path = SCRATCH / f"tornet_{year}.tar.gz"
    print(f"{year}: indiriliyor ...")
    tornet.download(tornet.tar_url(year), tar_path)
    print(f"{year}: işleniyor ...")
    tornet.process_year(year, tar_path, OUT, cfg["variables"], cfg["sweep"], cfg["frame"], catalog=cat)
    tar_path.unlink()               # sonraki yıl için Colab diskinde yer aç
    print(f"{year}: arşiv silindi, geçici klasörde {free_gb(SCRATCH):.0f} GB boş")

## Sonuçları katalogla karşılaştır

In [ ]:
for year in cfg["years"]:
    d = np.load(OUT / f"tornet_{year}.npz")
    X, y, names = d["X"], d["y"], d["filename"]
    expected = cat[cat["year"] == year]
    match = cat.set_index("filename").loc[names, "label"].to_numpy()
    print(f"{year}: X {X.shape} {X.dtype} | {len(y):,} örnek (katalog {len(expected):,}) | "
          f"{int(y.sum()):,} tornado | etiketler katalogla aynı: {(match == y).all()}")